# Task 6, round 5: the last −0.5% (`plan_t6.md`)
Public 8,542,010; tier 4.0 needs ≤ 8.5M. Public has moved 1.1–1.4× the validation change,
so the target is **≈ −0.4% on backtest A**, and negative on B too.

Everything is scored against the shipped model (127 leaves, × 1.07) on the **same 5 seeds**,
with features cached once per backtest, so comparisons are paired.

- **R1** diagnostics: where the error is (gap length, segments, data end)
- **R2** one scale per predicted-gap bucket
- **R3** similarity of the next book to the user's last reads (Tasks 2 / 4 lesson)
- **R4** cap predictions at the end of the data
- **R5** 10 seeds instead of 5

In [13]:
%load_ext autoreload
%autoreload 2

import time

import numpy as np
import pandas as pd
from sklearn.base import clone

from assignment_1.modules.models import (
    GapModel,
    GapRegressor,
    ItemSimilarity,
    LocalSimilarity,
    binary_cosine,
    split_last,
)
from assignment_1.modules.models.time.gaps import SECONDS_PER_DAY
from assignment_1.modules.tasks.task6 import TASK6_GAP_PARAMS, TASK6_REGRESSOR_PARAMS
from assignment_1.modules.util import clip_timestamps, load_csv

DATA_DIR = "../data"  # notebook runs from notebooks/
train = load_csv("task_6_train.csv", DATA_DIR)
trA, valA = split_last(train)
trB, valB = split_last(trA)
REG = TASK6_REGRESSOR_PARAMS["regressor"]
SCALE = TASK6_REGRESSOR_PARAMS["scale"]   # 1.07
SEEDS = tuple(range(TASK6_REGRESSOR_PARAMS["n_bag"]))   # 0..4, as shipped

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Cache features (shipped feature set) for both backtests
Training rows: each user's last gap in `tr`, features from the history before it.
Validation rows: the held-out gap, features from `tr`. Same as `GapRegressor.fit / predict`.

In [14]:
def cache(tr: pd.DataFrame, val: pd.DataFrame) -> dict:
    m = GapRegressor(base=GapModel(**TASK6_GAP_PARAMS), **TASK6_REGRESSOR_PARAMS)
    hist_b, last_b = split_last(tr)
    return {"hist": hist_b, "rows": last_b.reset_index(drop=True),
            "X": m.features(hist_b, last_b), "y": np.log1p(last_b["target"].to_numpy()),
            "vhist": tr, "vrows": val.reset_index(drop=True),
            "Xv": m.features(tr, val), "yv": val["target"].to_numpy()}


t0 = time.time()
C = {"A": cache(trA, valA), "B": cache(trB, valB)}
print(f"cached in {time.time() - t0:.0f}s; A train {C['A']['X'].shape}, val {C['A']['Xv'].shape}")

cached in 119s; A train (55403, 51), val (55403, 51)


In [15]:
def bag_pred(b: str, extra: tuple | None = None, seeds=SEEDS) -> np.ndarray:
    """Unscaled bagged prediction (seconds) on backtest b's validation rows.
    extra: (train columns, val columns) DataFrames appended to the cached features."""
    c = C[b]
    X, Xv = c["X"], c["Xv"]
    if extra is not None:
        X = pd.concat([X, extra[0].reset_index(drop=True)], axis=1)
        Xv = pd.concat([Xv, extra[1].reset_index(drop=True)], axis=1)
    logp = np.mean([clone(REG).set_params(random_state=s).fit(X, c["y"]).predict(Xv)
                    for s in seeds], axis=0)
    return np.expm1(logp)


def mae(p: np.ndarray, b: str) -> float:
    return float(np.mean(np.abs(p - C[b]["yv"])))


GRID = np.round(np.arange(0.90, 1.40, 0.005), 3)


def best_scale(p: np.ndarray, y: np.ndarray) -> float:
    return float(GRID[np.argmin([np.mean(np.abs(s * p - y)) for s in GRID])])


t0 = time.time()
P0 = {b: bag_pred(b) for b in "AB"}
REF = {b: mae(SCALE * P0[b], b) for b in "AB"}
print(f"shipped, 5 seeds × {SCALE}: A {REF['A']:,.0f}  B {REF['B']:,.0f}  ({time.time() - t0:.0f}s)")
print(f"best single scale: A {best_scale(P0['A'], C['A']['yv'])}, B {best_scale(P0['B'], C['B']['yv'])}")

results = []


def record(name: str, preds: dict[str, np.ndarray]) -> pd.DataFrame:
    """preds: backtest -> final (scaled) prediction."""
    row = {"variant": name}
    for b, p in preds.items():
        row[f"MAE {b}"] = mae(p, b)
        row[f"Δ% {b}"] = 100 * (row[f"MAE {b}"] / REF[b] - 1)
    results.append(row)
    print(f"{name:<40}", "  ".join(f"{b}: {100 * (mae(p, b) / REF[b] - 1):+.2f}%"
                                   for b, p in preds.items()))
    return pd.DataFrame(results).round(3)


record("shipped (reference)", {b: SCALE * P0[b] for b in "AB"})

shipped, 5 seeds × 1.07: A 7,062,989  B 7,058,508  (44s)
best single scale: A 1.075, B 1.05
shipped (reference)                      A: +0.00%  B: +0.00%


,variant,MAE A,Δ% A,MAE B,Δ% B
0,shipped (reference),7062989.474,0.0,7058507.93,0.0


## R1. Diagnostics
Per bucket: share of rows, share of total error, median log ratio log(pred / true)
(< 0: predicts too short), and the bucket's own best scale with the total gain it would give.
By **true** gap shows where the error is; by **predicted** gap is what R2 can act on.

In [16]:
EDGES_D = [0, 7, 30, 90, 365, np.inf]


def bucket_table(b: str, by: str) -> pd.DataFrame:
    y, p = C[b]["yv"], SCALE * P0[b]
    key = (y if by == "true" else p) / SECONDS_PER_DAY
    bucket = pd.cut(key, EDGES_D, right=False)
    err = np.abs(p - y)
    rows = []
    for lev in bucket.categories:
        m = (bucket == lev)
        if not m.any():
            continue
        s = best_scale(p[m], y[m])
        gain = np.abs(s * p[m] - y[m]).sum() - err[m].sum()
        rows.append({"bucket (days)": str(lev), "rows %": 100 * m.mean(),
                     "error %": 100 * err[m].sum() / err.sum(),
                     "mean |err| d": err[m].mean() / SECONDS_PER_DAY,
                     "median log(p/y)": np.median(np.log1p(p[m]) - np.log1p(y[m])),
                     "own best × (on top of 1.07)": s,
                     "Δ% total if used": 100 * gain / err.sum()})
    return pd.DataFrame(rows).round(3)


for b in "AB":
    print(f"backtest {b}, by TRUE gap")
    display(bucket_table(b, "true"))
for b in "AB":
    print(f"backtest {b}, by PREDICTED gap")
    display(bucket_table(b, "pred"))

backtest A, by TRUE gap


,bucket (days),rows %,error %,mean |err| d,median log(p/y),own best × (on top of 1.07),Δ% total if used
0,"[0.0, 7.0)",15.700,5.078,26.441,1.974,0.900,-0.553
1,"[7.0, 30.0)",23.744,6.977,24.020,0.544,0.900,-0.860
2,"[30.0, 90.0)",23.809,7.866,27.008,-0.200,0.900,-0.187
3,"[90.0, 365.0)",26.930,33.076,100.405,-0.796,1.255,-1.036
4,"[365.0, inf)",9.817,47.003,391.392,-1.233,1.085,-0.351


backtest B, by TRUE gap


,bucket (days),rows %,error %,mean |err| d,median log(p/y),own best × (on top of 1.07),Δ% total if used
0,"[0.0, 7.0)",16.871,5.736,27.778,1.989,0.900,-0.623
1,"[7.0, 30.0)",24.914,7.227,23.699,0.558,0.900,-0.900
2,"[30.0, 90.0)",24.087,8.215,27.863,-0.176,0.900,-0.275
3,"[90.0, 365.0)",25.589,31.521,100.635,-0.808,1.250,-0.950
4,"[365.0, inf)",8.539,47.300,452.525,-1.187,1.075,-0.271


backtest A, by PREDICTED gap


,bucket (days),rows %,error %,mean |err| d,median log(p/y),own best × (on top of 1.07),Δ% total if used
0,"[0.0, 7.0)",5.465,0.589,8.809,-0.033,1.055,-0.000
1,"[7.0, 30.0)",26.587,8.998,27.666,-0.063,1.065,-0.008
2,"[30.0, 90.0)",46.263,49.313,87.138,-0.155,1.170,-0.254
3,"[90.0, 365.0)",17.613,31.473,146.076,-0.065,1.035,-0.027
4,"[365.0, inf)",4.072,9.627,193.269,0.036,0.970,-0.074


backtest B, by PREDICTED gap


,bucket (days),rows %,error %,mean |err| d,median log(p/y),own best × (on top of 1.07),Δ% total if used
0,"[0.0, 7.0)",6.009,0.807,10.974,-0.110,1.155,-0.002
1,"[7.0, 30.0)",27.251,14.113,42.309,-0.055,1.050,-0.004
2,"[30.0, 90.0)",45.927,45.018,80.078,-0.046,1.050,-0.027
3,"[90.0, 365.0)",17.169,28.434,135.300,-0.003,1.000,0.000
4,"[365.0, inf)",3.644,11.628,260.685,0.042,0.960,-0.086


Segments and the data end. `to_end_d` = days from the user's last interaction to the end of
the data; no true gap can exceed it.

In [17]:
for b in "AB":
    Xv, y, p = C[b]["Xv"], C[b]["yv"], SCALE * P0[b]
    err = np.abs(p - y)
    to_end = Xv["to_end_d"].to_numpy() * SECONDS_PER_DAY
    print(f"backtest {b}: pred > to_end {np.mean(p > to_end):.2%} | true > to_end "
          f"{np.mean(y > to_end):.2%} (sanity, expect ~0)")
    segs = {
        "bound active (book appears after t_last)": Xv["bound_days"].to_numpy() > 0,
        "co-read delay defined": Xv["co_n"].fillna(0).to_numpy() > 0,
        "no co-read delay": Xv["co_n"].fillna(0).to_numpy() == 0,
        "next book has no other readers": Xv["it_n"].fillna(0).to_numpy() == 0,
    }
    display(pd.DataFrame({
        name: {"rows %": 100 * m.mean(), "error %": 100 * err[m].sum() / err.sum(),
               "mean |err| d": err[m].mean() / SECONDS_PER_DAY,
               "median log(p/y)": np.median(np.log1p(p[m]) - np.log1p(y[m]))}
        for name, m in segs.items()}).T.round(3))

backtest A: pred > to_end 0.03% | true > to_end 0.00% (sanity, expect ~0)


,rows %,error %,mean |err| d,median log(p/y)
bound active (book appears after t_last),16.981,23.889,115.005,0.005
co-read delay defined,71.467,65.510,74.934,-0.051
no co-read delay,28.533,34.490,98.815,-0.150
next book has no other readers,6.586,7.378,91.575,-0.179


backtest B: pred > to_end 0.00% | true > to_end 0.00% (sanity, expect ~0)


,rows %,error %,mean |err| d,median log(p/y)
bound active (book appears after t_last),16.243,23.312,117.249,0.016
co-read delay defined,70.049,64.682,75.436,-0.018
no co-read delay,29.951,35.318,96.335,-0.063
next book has no other readers,6.687,6.620,80.871,-0.087


## R2. One scale per predicted-gap bucket
Fit each bucket's scale on one backtest, relative to that backtest's best single scale, and
apply it to the other as `1.07 × relative`. This keeps the B → A drift that the single
1.07 already corrects. Both directions; keep only if both improve.

In [18]:
R2_EDGES = {"2 buckets (60 d)": [0, 60, np.inf], "3 buckets (30 / 120 d)": [0, 30, 120, np.inf]}


def bucket_scales(b: str, edges) -> np.ndarray:
    y, p = C[b]["yv"], P0[b]
    g = best_scale(p, y)
    k = np.digitize(p / SECONDS_PER_DAY, edges[1:-1])
    return np.array([best_scale(p[k == j], y[k == j]) / g for j in range(len(edges) - 1)])


def apply_scales(b: str, edges, rel: np.ndarray) -> np.ndarray:
    p = P0[b]
    k = np.digitize(p / SECONDS_PER_DAY, edges[1:-1])
    return SCALE * rel[k] * p


for name, edges in R2_EDGES.items():
    rel_B, rel_A = bucket_scales("B", edges), bucket_scales("A", edges)
    print(f"{name}: relative scales fit on B {rel_B.round(3)}, on A {rel_A.round(3)}")
    record(f"R2 {name}", {"A": apply_scales("A", edges, rel_B),
                          "B": apply_scales("B", edges, rel_A)})

2 buckets (60 d): relative scales fit on B [1.062 0.995], on A [1.126 0.986]
R2 2 buckets (60 d)                      A: -0.09%  B: -0.00%
3 buckets (30 / 120 d): relative scales fit on B [1.067 1.067 0.99 ], on A [1.065 1.163 0.972]
R2 3 buckets (30 / 120 d)                A: -0.27%  B: +0.07%


## R3. Similarity of the next book to the user's recent books
Cosine (fitted on the same history as the other features) between the next book and:
the whole history (`cos_sum / max / mean`) and the user's last 10 reads (`LocalSimilarity`
queried just after t_last: last 1 / 3 / 10 reads, reads within 1 / 7 / 30 days of t_last,
decay). EASE is out: 192k books.

In [19]:
def sim_features(hist: pd.DataFrame, rows: pd.DataFrame) -> pd.DataFrame:
    sim = ItemSimilarity(binary_cosine).fit(hist)
    loc = LocalSimilarity(binary_cosine, n_side=10).fit(hist, sim=sim)
    t_last = clip_timestamps(hist["timestamp"]).groupby(hist["user"]).max()
    t_q = rows["user"].map(t_last).fillna(0) + 1        # just after the last read
    return pd.concat([sim.features(rows["user"], rows["item"]),
                      loc.features(rows["user"], rows["item"], t_q)], axis=1)


t0 = time.time()
SIM = {b: (sim_features(C[b]["hist"], C[b]["rows"]), sim_features(C[b]["vhist"], C[b]["vrows"]))
       for b in "AB"}
print(f"similarity features in {time.time() - t0:.0f}s")
SIM["A"][1].describe().T[["mean", "50%", "max"]].round(3)

similarity features in 48s


,mean,50%,max
cos_sum,0.451,0.149,53.364
cos_max,0.113,0.060,0.913
cos_mean,0.031,0.014,0.678
loc_w1_sum,0.053,0.000,3.488
loc_w1_max,0.050,0.000,0.913
loc_w7_sum,0.062,0.000,6.468
loc_w7_max,0.053,0.000,0.913
loc_w30_sum,0.084,0.012,6.468
loc_w30_max,0.061,0.011,0.913
loc_k1_mean,0.049,0.000,0.913


In [20]:
P3 = {b: bag_pred(b, extra=SIM[b]) for b in "AB"}
record("R3 + similarity (× 1.07)", {b: SCALE * P3[b] for b in "AB"})
print("best single scale with similarity:",
      {b: best_scale(P3[b], C[b]["yv"]) for b in "AB"})

R3 + similarity (× 1.07)                 A: -0.17%  B: -0.34%
best single scale with similarity: {'A': 1.075, 'B': 1.045}


Only the last-reads part (drop the whole-history cosine), in case the two overlap.

In [21]:
LOC_ONLY = {b: tuple(f.filter(like="loc_") for f in SIM[b]) for b in "AB"}
P3l = {b: bag_pred(b, extra=LOC_ONLY[b]) for b in "AB"}
record("R3 + last-reads similarity only", {b: SCALE * P3l[b] for b in "AB"})

R3 + last-reads similarity only          A: -0.20%  B: -0.41%


,variant,MAE A,Δ% A,MAE B,Δ% B
0,shipped (reference),7062989.474,0.000,7058507.930,0.000
1,R2 2 buckets (60 d),7056667.919,-0.090,7058361.715,-0.002
2,R2 3 buckets (30 / 120 d),7043946.780,-0.270,7063195.840,0.066
3,R3 + similarity (× 1.07),7051080.235,-0.169,7034261.259,-0.344
4,R3 + last-reads similarity only,7048623.851,-0.203,7029298.000,-0.414


## R4. Cap at the end of the data

In [22]:
def capped(p: np.ndarray, b: str) -> np.ndarray:
    # fmin: 27 users have no in-range timestamp, so to_end_d is NaN -> keep the prediction
    return np.fmin(p, C[b]["Xv"]["to_end_d"].to_numpy() * SECONDS_PER_DAY)


record("R4 cap at data end", {b: capped(SCALE * P0[b], b) for b in "AB"})
record("R3 + R4", {b: capped(SCALE * P3[b], b) for b in "AB"})
for b in "AB":
    to_end = C[b]["Xv"]["to_end_d"].to_numpy() * SECONDS_PER_DAY
    print(f"{b}: capped {np.mean(SCALE * P0[b] > to_end):.2%} of rows")

R4 cap at data end                       A: -0.01%  B: -0.00%
R3 + R4                                  A: -0.18%  B: -0.34%
A: capped 0.03% of rows
B: capped 0.00% of rows


## R6. Combine: last-reads similarity + cap, and its own scale
Scale picked on B relative to B's best single scale, applied to A as 1.07 × relative (and the reverse),
as in R2: new features can shift calibration.

In [23]:
record("R3 last-reads + R4 cap", {b: capped(SCALE * P3l[b], b) for b in "AB"})

g0 = {b: best_scale(P0[b], C[b]["yv"]) for b in "AB"}
g3 = {b: best_scale(P3l[b], C[b]["yv"]) for b in "AB"}
print("best single scale  shipped:", g0, " last-reads:", g3)
rel = {"A": g3["B"] / g0["B"], "B": g3["A"] / g0["A"]}   # fit on the other backtest
record("R3 last-reads, rescaled", {b: SCALE * rel[b] * P3l[b] for b in "AB"})
record("R3 last-reads, rescaled + R4 cap", {b: capped(SCALE * rel[b] * P3l[b], b) for b in "AB"})


R3 last-reads + R4 cap                   A: -0.22%  B: -0.41%
best single scale  shipped: {'A': 1.075, 'B': 1.05}  last-reads: {'A': 1.075, 'B': 1.05}
R3 last-reads, rescaled                  A: -0.20%  B: -0.41%
R3 last-reads, rescaled + R4 cap         A: -0.22%  B: -0.41%


,variant,MAE A,Δ% A,MAE B,Δ% B
0,shipped (reference),7062989.474,0.000,7058507.930,0.000
1,R2 2 buckets (60 d),7056667.919,-0.090,7058361.715,-0.002
2,R2 3 buckets (30 / 120 d),7043946.780,-0.270,7063195.840,0.066
3,R3 + similarity (× 1.07),7051080.235,-0.169,7034261.259,-0.344
4,R3 + last-reads similarity only,7048623.851,-0.203,7029298.000,-0.414
5,R4 cap at data end,7062162.432,-0.012,7058463.151,-0.001
6,R3 + R4,7050226.275,-0.181,7034201.117,-0.344
7,R3 last-reads + R4 cap,7047722.193,-0.216,7029231.786,-0.415
8,"R3 last-reads, rescaled",7048623.851,-0.203,7029298.000,-0.414
9,"R3 last-reads, rescaled + R4 cap",7047722.193,-0.216,7029231.786,-0.415


## R5. 10 seeds (set `FINAL_EXTRA` to the best of R3: `SIM`, `LOC_ONLY` or `None`)

In [24]:
FINAL_EXTRA = LOC_ONLY   # best of R3
P5 = {b: bag_pred(b, extra=None if FINAL_EXTRA is None else FINAL_EXTRA[b], seeds=range(10))
      for b in "AB"}
record("R5 final features, 10 seeds", {b: SCALE * P5[b] for b in "AB"})

R5 final features, 10 seeds              A: -0.20%  B: -0.40%


,variant,MAE A,Δ% A,MAE B,Δ% B
0,shipped (reference),7062989.474,0.000,7058507.930,0.000
1,R2 2 buckets (60 d),7056667.919,-0.090,7058361.715,-0.002
2,R2 3 buckets (30 / 120 d),7043946.780,-0.270,7063195.840,0.066
3,R3 + similarity (× 1.07),7051080.235,-0.169,7034261.259,-0.344
4,R3 + last-reads similarity only,7048623.851,-0.203,7029298.000,-0.414
5,R4 cap at data end,7062162.432,-0.012,7058463.151,-0.001
6,R3 + R4,7050226.275,-0.181,7034201.117,-0.344
7,R3 last-reads + R4 cap,7047722.193,-0.216,7029231.786,-0.415
8,"R3 last-reads, rescaled",7048623.851,-0.203,7029298.000,-0.414
9,"R3 last-reads, rescaled + R4 cap",7047722.193,-0.216,7029231.786,-0.415


## Summary
Ship if a variant is negative on both backtests and ≤ −0.4% on A.

In [25]:
pd.DataFrame(results).round(3)

,variant,MAE A,Δ% A,MAE B,Δ% B
0,shipped (reference),7062989.474,0.000,7058507.930,0.000
1,R2 2 buckets (60 d),7056667.919,-0.090,7058361.715,-0.002
2,R2 3 buckets (30 / 120 d),7043946.780,-0.270,7063195.840,0.066
3,R3 + similarity (× 1.07),7051080.235,-0.169,7034261.259,-0.344
4,R3 + last-reads similarity only,7048623.851,-0.203,7029298.000,-0.414
5,R4 cap at data end,7062162.432,-0.012,7058463.151,-0.001
6,R3 + R4,7050226.275,-0.181,7034201.117,-0.344
7,R3 last-reads + R4 cap,7047722.193,-0.216,7029231.786,-0.415
8,"R3 last-reads, rescaled",7048623.851,-0.203,7029298.000,-0.414
9,"R3 last-reads, rescaled + R4 cap",7047722.193,-0.216,7029231.786,-0.415
